# Shallow Learning — classification + heatmap + comparison

Consolidates the logic of notebooks 07-10 from `Grid-Finding/` applied to the multi-city `all_cities_combined.csv` dataset.

**Structure:**
1. Configuration (identical to `02_DimReduction_OSM.ipynb`)
2. Load + filters + balancing
3. EDA: class distribution, per-feature boxplots, correlation
4. Pairplot
5. ML — Logistic Regression, Random Forest, XGBoost
6. Feature Importance
7. 3-model accuracy bar chart + ROC curves
8. Geospatial heatmap (one subplot per city)
9. Model comparison and per-city accuracy

**Prediction output** at `ramon/csv/predictions_<source>.csv`. Plots saved to `outputs/ShallowLearning/`.

## 1. Imports + configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pathlib

pd.set_option('display.float_format', lambda x: '%.3f' % x)
sns.set_style('darkgrid')

### Analysis configuration

**Parameters live in `config.py`** (same folder). Edit that file to change `SOURCE`, `CITIES`, `CITY_GROUP`, `BALANCE_METHOD`, `FEATURES`, `EXCLUDE_FEATURES`, paths, etc.

Both this notebook and `02_DimReduction_OSM.ipynb` read from the same `config.py`. After editing, do **Restart Kernel + Run All**.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path(r"E:\IAAC Local GIT Repositories\OSMnx-data-scraper\2026.05.25_ShallowLearning-Dimensionality_ramón")))

import importlib, config
importlib.reload(config)            # re-read config.py if you edited it without restart
from config import *

# This notebook saves outputs in the ShallowLearning subfolder
PLOTS_DIR = PLOTS_DIR_SHALLOW

print(f"SOURCE        = {SOURCE!r}")
print(f"CITIES        = {CITIES}")
print(f"CITY_GROUP    = {CITY_GROUP}")
print(f"BALANCE       = {BALANCE_METHOD}")
print(f"FEATURES      = {FEATURES}")
print(f"EXCLUDE       = {EXCLUDE_FEATURES}")
print(f"RANDOM_STATE  = {RANDOM_STATE}")
print(f"TEST_SIZE     = {TEST_SIZE}")
print(f"PLOTS_DIR     = {PLOTS_DIR}")

## 2. Load dataset + filters + balancing

In [ ]:
data_raw = pd.read_csv(CSV_PATH)
print(f"Original shape: {data_raw.shape}")
data_raw.head(3)

In [ ]:
selected_cities = resolve_cities(data_raw['city'].unique())
print(f"Selected cities: {selected_cities}")

data = data_raw[data_raw['city'].isin(selected_cities)].copy()
data = data[data['zone_type'].isin(['Commercial', 'Residential'])].copy()
data = data.reset_index(drop=True)

print(f"\nShape after city + class filters: {data.shape}")
print(data['zone_type'].value_counts())

In [ ]:
# --- Balancing ---
if BALANCE_METHOD == 'undersample':
    n_min = data['zone_type'].value_counts().min()
    pieces = [
        data[data['zone_type'] == cls].sample(n=n_min, random_state=RANDOM_STATE)
        for cls in data['zone_type'].unique()
    ]
    data = pd.concat(pieces).reset_index(drop=True)
    print(f"After undersample: {len(data)} rows")
elif BALANCE_METHOD == 'none':
    print('No balancing applied')
else:
    raise ValueError(f"Unknown BALANCE_METHOD: {BALANCE_METHOD!r}")

print('\nFinal distribution:')
print(data['zone_type'].value_counts())

## 3. Feature selection

In [ ]:
feature_cols = resolve_features()

print(f"SOURCE={SOURCE!r}, features ({len(feature_cols)}):")
for f in feature_cols:
    print(f"  - {f}")

data_num = data[feature_cols].copy()
data_num = data_num.fillna(data_num.median(numeric_only=True))
print(f"\nShape of data_num: {data_num.shape}")

## 4. EDA — Distribution, boxplots, correlation

In [ ]:
pathlib.Path(PLOTS_DIR).mkdir(parents=True, exist_ok=True)

# Countplot of zone_type
fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(data=data, x='zone_type', ax=ax)
ax.set_title('zone_type distribution')
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/01_countplot_{SOURCE}.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Boxplots per feature by class
n_feats = len(feature_cols)
ncols = min(3, n_feats)
nrows = int(np.ceil(n_feats / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*5, nrows*4))
axes = np.atleast_1d(axes).flatten()
fig.suptitle('Feature distribution by zone_type', fontsize=14)

for ax, feat in zip(axes, feature_cols):
    sns.boxplot(data=data, x='zone_type', y=feat, ax=ax)
    ax.set_title(feat)
for ax in axes[n_feats:]:
    ax.axis('off')

plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/02_boxplots_{SOURCE}.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation
plt.figure(figsize=(10, 8))
sns.heatmap(data_num.corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, vmin=-1, vmax=1)
plt.title(f'Feature correlation ({SOURCE})')
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/03_corr_{SOURCE}.png", dpi=150, bbox_inches='tight')
plt.show()

## 5. Pairplot (notebook 10)

In [ ]:
pairplot_n = min(2000, len(data_num))
pairplot_idx = data_num.sample(n=pairplot_n, random_state=RANDOM_STATE).index
sample_for_pairplot = pd.concat([
    data_num.loc[pairplot_idx],
    data.loc[pairplot_idx, ['zone_type']]
], axis=1)
g = sns.pairplot(sample_for_pairplot, hue='zone_type',
                 plot_kws={'alpha': 0.4, 's': 12})
g.fig.suptitle(f'Pairplot features × zone_type ({SOURCE})', y=1.02)
g.savefig(f"{PLOTS_DIR}/04_pairplot_{SOURCE}.png", dpi=120, bbox_inches='tight')
plt.show()

## 6. Preprocessing + train/test split

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score

X = data_num.values
encoder = LabelEncoder()
y = encoder.fit_transform(data['zone_type'])
class_names = list(encoder.classes_)
print(f"Classes: {class_names}")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X_scaled, y, np.arange(len(y)),
    test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

print(f"Train: {X_train.shape[0]}  Test: {X_test.shape[0]}")
print(f"Train balance: {np.bincount(y_train)}")
print(f"Test balance:  {np.bincount(y_test)}")

## 7. Models — LogReg, Random Forest, XGBoost (notebook 07)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from mlxtend.plotting import plot_confusion_matrix

models = {
    'LogReg':  LogisticRegression(max_iter=1000, class_weight='balanced',
                                  random_state=RANDOM_STATE),
    'RandomForest': RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                           random_state=RANDOM_STATE),
    'XGBoost': XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                             eval_metric='logloss', random_state=RANDOM_STATE),
}

results = {}
predictions_dict = {}

for name, model in models.items():
    print(f"\n=== {name} ===")
    cv = cross_val_score(model, X_train, y_train, cv=5)
    print(f"CV: {cv.mean():.3f} ± {cv.std():.3f}")

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    print(f"Test accuracy: {acc:.3f}  |  F1: {f1:.3f}")
    print(classification_report(y_test, y_pred, target_names=class_names))

    results[name] = {'cv_mean': cv.mean(), 'cv_std': cv.std(),
                     'test_acc': acc, 'test_f1': f1}
    predictions_dict[name] = y_pred

    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plot_confusion_matrix(conf_mat=cm, colorbar=True,
                                    show_absolute=True, show_normed=True,
                                    class_names=class_names)
    plt.title(f'{name} — Confusion Matrix')
    plt.tight_layout()
    plt.savefig(f"{PLOTS_DIR}/05_cm_{name}_{SOURCE}.png", dpi=150, bbox_inches='tight')
    plt.show()

## 8. Feature Importance (Random Forest)

In [ ]:
rf = models['RandomForest']
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values()

fig, ax = plt.subplots(figsize=(10, max(4, 0.4*len(importances))))
importances.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title(f'Random Forest — Feature Importance ({SOURCE})')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/06_feat_importance_{SOURCE}.png", dpi=150, bbox_inches='tight')
plt.show()

print('Top features:')
print(importances.sort_values(ascending=False).head(10).to_string())

## 9. Geospatial heatmap (notebook 08)

Scatter map over lat/lon colored by the best model's prediction. When several cities are selected we draw one subplot per city at its own scale.

In [ ]:
# Generate predictions for ALL rows (not just test) using the best model
best_model_name = max(results, key=lambda k: results[k]['test_f1'])
best_model = models[best_model_name]
print(f"Best model: {best_model_name}")

y_all_pred = best_model.predict(X_scaled)
y_all_proba = best_model.predict_proba(X_scaled)
comm_idx = list(encoder.classes_).index('Commercial')

data_pred = data.copy()
data_pred['predicted_zone'] = encoder.inverse_transform(y_all_pred)
data_pred['prob_commercial'] = y_all_proba[:, comm_idx].round(4)

# Export predictions
pathlib.Path(EXPORT_DIR).mkdir(parents=True, exist_ok=True)
pred_path = pathlib.Path(EXPORT_DIR) / f"predictions_{SOURCE}.csv"
data_pred[['city','cell_id','cell_lat','cell_lon','zone_type',
           'predicted_zone','prob_commercial']].to_csv(pred_path, index=False)
print(f"Saved: {pred_path}")

In [ ]:
cities_present = sorted(data_pred['city'].unique())
n_cities = len(cities_present)
ncols = min(3, n_cities)
nrows = int(np.ceil(n_cities / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*5, nrows*5), squeeze=False)
fig.suptitle(f'Heatmap prob_commercial — {best_model_name} ({SOURCE})', fontsize=14)

for ax, city in zip(axes.flatten(), cities_present):
    sub = data_pred[data_pred['city'] == city]
    sc = ax.scatter(sub['cell_lon'], sub['cell_lat'],
                    c=sub['prob_commercial'], cmap='RdBu_r',
                    vmin=0, vmax=1, s=12, alpha=0.8, edgecolor='none')
    ax.set_title(f'{city}  (n={len(sub)})')
    ax.set_xlabel('lon'); ax.set_ylabel('lat')
    ax.set_aspect('equal')
    plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04, label='P(Commercial)')

for ax in axes.flatten()[n_cities:]:
    ax.axis('off')

plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/07_heatmap_{SOURCE}.png", dpi=150, bbox_inches='tight')
plt.show()

## 10. Comparison (notebook 09)

Three views:
1. Accuracy and F1 per model
2. Best model's accuracy broken down by city
3. Cross-tab of real `zone_type` × `predicted_zone`

In [ ]:
# Model comparison
res_df = pd.DataFrame(results).T
print(res_df.round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
res_df[['cv_mean','test_acc']].plot(kind='bar', ax=axes[0])
axes[0].set_title('Accuracy: CV vs Test')
axes[0].set_ylim(0, 1)
axes[0].axhline(0.5, color='red', linestyle='--', alpha=0.5, label='random')
axes[0].legend()
axes[0].set_xticklabels(res_df.index, rotation=0)

res_df[['test_f1']].plot(kind='bar', ax=axes[1], color='orange')
axes[1].set_title('F1 weighted (test)')
axes[1].set_ylim(0, 1)
axes[1].set_xticklabels(res_df.index, rotation=0)

plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/08_model_comparison_{SOURCE}.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Best model's accuracy by city (on the FULL dataset, not just test)
data_pred['correct'] = data_pred['zone_type'] == data_pred['predicted_zone']
per_city = data_pred.groupby('city').agg(
    n=('correct', 'size'),
    accuracy=('correct', 'mean'),
).sort_values('accuracy', ascending=False)
print(per_city.round(3).to_string())

fig, ax = plt.subplots(figsize=(8, max(3, 0.5*len(per_city))))
per_city['accuracy'].plot(kind='barh', ax=ax, color='steelblue')
ax.axvline(0.5, color='red', linestyle='--', alpha=0.5, label='random')
ax.set_xlim(0, 1)
ax.set_title(f'Accuracy by city — {best_model_name} ({SOURCE})')
ax.set_xlabel('Accuracy')
ax.legend()
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/09_accuracy_by_city_{SOURCE}.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Real vs predicted cross-tab
ct = pd.crosstab(data_pred['zone_type'], data_pred['predicted_zone'],
                 margins=True, margins_name='Total')
print('Real × predicted cross-tab (all data):')
print(ct.to_string())

## Summary

Outputs generated under `outputs/`:
- `01_countplot_<SOURCE>.png`
- `02_boxplots_<SOURCE>.png`
- `03_corr_<SOURCE>.png`
- `04_pairplot_<SOURCE>.png`
- `05_cm_<Model>_<SOURCE>.png` (3 confusion matrices)
- `06_feat_importance_<SOURCE>.png`
- `07_heatmap_<SOURCE>.png`
- `08_model_comparison_<SOURCE>.png`
- `09_accuracy_by_city_<SOURCE>.png`
- `10_accuracy_3models_<SOURCE>.png`
- `11_roc_3models_<SOURCE>.png`

Predictions in `ramon/csv/predictions_<SOURCE>.csv`.

To compare property vs OSM, run this notebook twice changing `SOURCE` and compare the outputs.